In [ ]:
#https://search.worldbank.org/api/v3/projects  project - 27848

#https://search.worldbank.org/api/v3/wds       document - 594578


In [ ]:
import requests
import json
import pandas as pd
import wbgapi as wb
import time
from function import get_document, get_project, get_document_bulk

projects_url = 'https://search.worldbank.org/api/v3/projects'
documents_url = 'https://search.worldbank.org/api/v3/wds'

In [2]:
all_projects = {}
os = 0
total = 27848

while os < total:
    data = get_project(projects_url, rows=1000, os=os)
    if data is None:
        print(f"Skipping os={os} after failed retries")
        os += 1000
        continue
    projects = data.get('projects', {})
    all_projects.update(projects)
    print(f"Fetched {len(all_projects)} / {total}")
    os += 1000
    time.sleep(1)  # be polite to the API

print(f"Done: {len(all_projects)} projects")



Fetched 1000 / 27848


KeyboardInterrupt: 

In [3]:
project_df = pd.json_normalize(all_projects.values())
project_df.drop(columns=['sectorcode', 'major_sector_name', 'major_sector_code'], inplace=True)

### call projects w/o project api call

In [3]:
project_df = pd.read_excel('look_wb_api_full.xlsx')
project_ids = project_df['proj_id'].tolist()


In [4]:
pythonproject_df = pd.read_excel('look_wb_api_full.xlsx')
project_ids = project_df['proj_id'].tolist()

# bulk fetch all documents
all_documents = {}
failed_offsets = []
os = 0
total = 594578

while os < total:
    data = get_document_bulk(rows=1000, os=os)
    if data is None:
        failed_offsets.append(os)
        os += 1000
        continue
    
    docs = data.get('documents', {})
    if not docs:
        break
    
    project_docs = {k: v for k, v in docs.items() if v.get('projectid')}
    all_documents.update(project_docs)
    
    print(f"os={os}, kept {len(project_docs)}/1000, total {len(all_documents)}")
    os += 1000
    time.sleep(0.5)




os=0, kept 929/1000, total 929
os=1000, kept 927/1000, total 1856
os=2000, kept 917/1000, total 2773
os=3000, kept 969/1000, total 3742
os=4000, kept 952/1000, total 4694
os=5000, kept 956/1000, total 5649
os=6000, kept 952/1000, total 6600
os=7000, kept 944/1000, total 7544
os=8000, kept 947/1000, total 8491
os=9000, kept 895/1000, total 9385
os=10000, kept 895/1000, total 10280
os=11000, kept 899/1000, total 11179
os=12000, kept 923/1000, total 12102
os=13000, kept 916/1000, total 13018
os=14000, kept 905/1000, total 13922
os=15000, kept 861/1000, total 14783
os=16000, kept 269/1000, total 15052
os=17000, kept 555/1000, total 15607
os=18000, kept 936/1000, total 16543
os=19000, kept 949/1000, total 17490
os=20000, kept 930/1000, total 18420
os=21000, kept 963/1000, total 19383
os=22000, kept 963/1000, total 20346
os=23000, kept 929/1000, total 21275
os=24000, kept 942/1000, total 22217
os=25000, kept 948/1000, total 23165
os=26000, kept 917/1000, total 24082
os=27000, kept 925/1000, 

KeyboardInterrupt: 

## for later run

In [17]:
# resume main loop from where it stopped
os = resume_os
total = 594578

while os < total:
    data = get_document_bulk(rows=1000, os=os)
    if data is None:
        failed_offsets.append(os)
        os += 1000
        continue
    
    docs = data.get('documents', {})
    if not docs:
        break
    
    project_docs = {k: v for k, v in docs.items() if v.get('projectid')}
    all_documents.update(project_docs)
    
    print(f"os={os}, kept {len(project_docs)}/1000, total {len(all_documents)}")
    os += 1000
    time.sleep(0.5)

# checkpoint after main loop
with open('all_documents_checkpoint.json', 'w') as f:
    json.dump(all_documents, f)

# THEN retry failed offsets
print(f"\nRetrying {len(failed_offsets)} failed offsets...")
still_failed_offsets = []

for failed_os in failed_offsets:
    data = get_document_bulk(rows=1000, os=failed_os)
    if data is None:
        still_failed_offsets.append(failed_os)
        continue
    docs = data.get('documents', {})
    project_docs = {k: v for k, v in docs.items() if v.get('projectid')}
    all_documents.update(project_docs)
    print(f"Recovered os={failed_os}: {len(project_docs)} docs")
    time.sleep(1)

# THEN retry missing project ids
doc_project_ids = set(v.get('projectid') for v in all_documents.values() if v.get('projectid'))
missing_project_ids = [pid for pid in project_ids if pid not in doc_project_ids]

print(f"\nRetrying {len(missing_project_ids)} projects with no docs...")
for project_id in missing_project_ids:
    data = get_document(project_id)
    if data is None:
        continue
    docs = data.get('documents', {})
    if docs:
        all_documents.update(docs)
    time.sleep(1)

# final summary
doc_project_ids = set(v.get('projectid') for v in all_documents.values() if v.get('projectid'))
still_missing = [pid for pid in project_ids if pid not in doc_project_ids]

print(f"\nSummary:")
print(f"Total docs fetched: {len(all_documents)}")
print(f"Failed offsets: {still_failed_offsets}")
print(f"Projects still with no docs: {len(still_missing)}")

with open('failed_offsets.json', 'w') as f:
    json.dump(still_failed_offsets, f)
with open('missing_project_ids.json', 'w') as f:
    json.dump(still_missing, f)

Error 400 at os=101000, attempt 1/3, waiting 1s
Error 400 at os=101000, attempt 2/3, waiting 2s
Error 400 at os=101000, attempt 3/3, waiting 4s
Error 400 at os=102000, attempt 1/3, waiting 1s
Error 400 at os=102000, attempt 2/3, waiting 2s
Error 400 at os=102000, attempt 3/3, waiting 4s
Error 400 at os=103000, attempt 1/3, waiting 1s
Error 400 at os=103000, attempt 2/3, waiting 2s
Error 400 at os=103000, attempt 3/3, waiting 4s
Error 400 at os=104000, attempt 1/3, waiting 1s
Error 400 at os=104000, attempt 2/3, waiting 2s
Error 400 at os=104000, attempt 3/3, waiting 4s
Error 400 at os=105000, attempt 1/3, waiting 1s
Error 400 at os=105000, attempt 2/3, waiting 2s
Error 400 at os=105000, attempt 3/3, waiting 4s
Error 400 at os=106000, attempt 1/3, waiting 1s
Error 400 at os=106000, attempt 2/3, waiting 2s
Error 400 at os=106000, attempt 3/3, waiting 4s
Error 400 at os=107000, attempt 1/3, waiting 1s
Error 400 at os=107000, attempt 2/3, waiting 2s
Error 400 at os=107000, attempt 3/3, wai

KeyboardInterrupt: 

In [16]:
# find which project_ids have no docs yet
doc_project_ids = set(v.get('projectid') for v in all_documents.values() if v.get('projectid'))
missing_project_ids = [pid for pid in project_ids if pid not in doc_project_ids]

print(f"{len(missing_project_ids)} projects still need docs")

# fetch docs for missing projects
still_missing = []

for i, project_id in enumerate(missing_project_ids):
    data = get_document(project_id)
    if data is None:
        still_missing.append(project_id)
        continue
    
    docs = data.get('documents', {})
    if docs:
        all_documents.update(docs)
    else:
        still_missing.append(project_id)  # project genuinely has no docs
    
    if i % 10 == 0:
        print(f"Progress: {i}/{len(missing_project_ids)}, total docs {len(all_documents)}")
    
    time.sleep(0.5)

# save checkpoint
with open('all_documents_checkpoint.json', 'w') as f:
    json.dump(all_documents, f)

with open('still_missing_project_ids.json', 'w') as f:
    json.dump(still_missing, f)

print(f"\nSummary:")
print(f"Total docs: {len(all_documents)}")
print(f"Projects with no docs after retry: {len(still_missing)}")

23758 projects still need docs
Progress: 0/23758, total docs 91109
Progress: 10/23758, total docs 91109
Progress: 20/23758, total docs 91109


KeyboardInterrupt: 

In [ ]:
# get project ids that are in all_documents
doc_project_ids = set(v.get('projectid') for v in all_documents.values() if v.get('projectid'))

# find project ids with no documents fetched
missing_project_ids = [pid for pid in project_ids if pid not in doc_project_ids]

print(f"Projects with no documents: {len(missing_project_ids)}")
print(missing_project_ids[:10])  # preview first 10

In [ ]:
document_df = pd.json_normalize(all_documents.values())
document_df.to_excel('look_wb_api_documents.xlsx', index=False)

In [18]:
def get_document_bulk_dated(year, os=0, retries=3):
    documents_url = 'https://search.worldbank.org/api/v3/wds'
    url = f"{documents_url}?format=json&rows=1000&os={os}&strdate={year}-01-01&enddate={year}-12-31"
    for attempt in range(retries):
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            wait = 2 ** attempt
            print(f'Error {response.status_code} year={year} os={os}, attempt {attempt+1}/{retries}, waiting {wait}s')
            time.sleep(wait)
    return None

In [19]:
# fetch by year - each year will have well under 100k docs
for year in range(1946, 2026):
    os = 0
    while True:
        data = get_document_bulk_dated(year, os=os)
        if data is None or not data.get('documents'):
            break
        docs = data.get('documents', {})
        project_docs = {k: v for k, v in docs.items() if v.get('projectid')}
        all_documents.update(project_docs)
        total_year = int(data.get('total', 0))
        print(f"Year {year}, os={os}/{total_year}, total docs {len(all_documents)}")
        if os + 1000 >= total_year or os + 1000 >= 100000:
            break
        os += 1000
        time.sleep(0.5)

Year 1946, os=0/58, total docs 91109
Year 1947, os=0/161, total docs 91126
Error 500 year=1948 os=0, attempt 1/3, waiting 1s
Error 500 year=1948 os=0, attempt 2/3, waiting 2s
Year 1948, os=0/264, total docs 91145
Year 1949, os=0/245, total docs 91196
Year 1950, os=0/262, total docs 91281
Year 1951, os=0/249, total docs 91345
Year 1952, os=0/232, total docs 91410
Year 1953, os=0/216, total docs 91496
Year 1954, os=0/184, total docs 91569
Year 1955, os=0/289, total docs 91713
Year 1956, os=0/276, total docs 91848
Error 500 year=1957 os=0, attempt 1/3, waiting 1s
Year 1957, os=0/287, total docs 91978
Error 500 year=1958 os=0, attempt 1/3, waiting 1s
Error 500 year=1958 os=0, attempt 2/3, waiting 2s
Year 1958, os=0/351, total docs 92148
Year 1959, os=0/327, total docs 92303
Year 1960, os=0/321, total docs 92450
Year 1961, os=0/477, total docs 92701
Year 1962, os=0/374, total docs 92854
Year 1963, os=0/478, total docs 93076
Error 500 year=1964 os=0, attempt 1/3, waiting 1s
Year 1964, os=0/5

In [22]:
document_df = pd.json_normalize(all_documents.values())


In [ ]:
document_df.to_excel('all_documents_final.xlsx', index=False)

In [30]:
project_df[~project_df['id'].isin(document_df['projectid'])]

,id,proj_id,countryshortname,boardapprovaldate,curr_ibrd_commitment,grantamt,totalamt,regionname,lendprojectcost,borrower,...,idacommamt,project_name,projid_id_display,closingdate,sectorcode,major_sectors,major_sector_name,major_sector_code,theme_list,envassesmentcategorycode
0,P514733,P514733,Uzbekistan,2027-09-15T00:00:00Z,100000000.0,200000000.0,100000000.0,Europe and Central Asia,350249984.0,Republic of Uzbekistan,...,0.0,Uzbekistan Modernization of Electricity Distri...,P514733,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,P515269,P515269,Pakistan,2026-09-24T00:00:00Z,100000000.0,0.0,200000000.0,"Middle East, North Africa, Afghanistan, and Pa...",478249984.0,"Ministry of Economic Affairs, Government of Pa...",...,100000000.0,Pakistan Public Resources for Inclusive Develo...,P515269,2031-10-01,NaN,NaN,NaN,NaN,NaN,NaN
59,P181524,P181524,India,2026-08-25T00:00:00Z,420000000.0,0.0,420000000.0,South Asia,638.0,"Ministry of Finance, Department of Economic Af...",...,0.0,Second Dam Rehabilitation and Improvement Proj...,P181524,2027-12-31,NaN,NaN,NaN,NaN,NaN,NaN
122,P512523,P512523,Grenada,2026-07-02T00:00:00Z,0.0,0.0,40000000.0,Latin America and Caribbean,NaN,Grenada,...,40000000.0,Grenada - First competitiveness and resilience...,P512523,2029-07-27,NaN,NaN,NaN,NaN,NaN,NaN
134,P513901,P513901,Uzbekistan,2026-06-22T00:00:00Z,50000000.0,0.0,150000000.0,Europe and Central Asia,150000000.0,The Republic of Uzbekistan,...,100000000.0,Uzbekistan Risk Mitigation Facility,P513901,2031-06-30,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27840,P002599,P002599,Sudan,NaN,0.0,0.0,5000000.0,Eastern and Southern Africa,5000000.0,NaN,...,5000000.0,GAS EGR,P002599,NaN,GS,"[{'major_sector': {'major_sector_code': 'GX', ...",(Historic)Oil & Gas,GX,NaN,NaN
27841,P002085,P002085,Nigeria,NaN,55000000.0,0.0,55000000.0,Western and Central Africa,0.0,NaN,...,0.0,W/S III-(CROSS RIVER,P002085,NaN,WU,"[{'major_sector': {'major_sector_code': 'WX', ...","Water, Sanitation and Waste Management",WX,NaN,NaN
27842,P001695,P001695,Mali,NaN,0.0,0.0,8300000.0,Western and Central Africa,0.0,NaN,...,8300000.0,HIGHWAYS SUPPLEMENT,P001695,NaN,TH,"[{'major_sector': {'major_sector_code': 'TX', ...",Transportation,TX,NaN,NaN
27843,P001694,P001694,Mali,NaN,0.0,0.0,2600000.0,Western and Central Africa,0.0,NaN,...,2600000.0,MOPTI RICE SUPPLEMEN,P001694,NaN,AP,"[{'major_sector': {'major_sector_code': 'AX', ...","Agriculture, Fishing and Forestry",AX,NaN,NaN


In [29]:
document_df[~document_df['projectid'].isin(project_df['id'])]

,id,last_modified_date,admreg,count,docty,projn,subsc,theme,prdln,seccl,...,geo_regions.15.geo_region,sectr.29.sector,sectr.30.sector,sectr.31.sector,sectr.32.sector,sectr.33.sector,sectr.34.sector,sectr.35.sector,sectr.36.sector,sectr.37.sector
4,40093824,2026-04-02T23:05:08Z,Eastern and Southern Africa,Zambia,Country Climate and Development Report,ZM-Zambia Country Climate and Development Repo...,FY17 - Other Public Administration,"FY17 - Climate change,FY17 - Economic Growth a...",Advisory Services & Analytics,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,40095203,2026-04-02T23:05:08Z,Eastern and Southern Africa,"Congo, Democratic Republic of",Project Information Document,ZR-Investment Facilitation Project -- P512137,NaN,NaN,IBRD/IDA,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,40095240,2026-04-02T23:05:08Z,Latin America and Caribbean,Caribbean,Procurement Plan,6R-Project Preparation; Caribbean Water Securi...,NaN,NaN,Preparation Facilities,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,40095065,2026-04-02T23:05:08Z,Latin America and Caribbean,Colombia,Environmental and Social Review Summary,CO-Support to the Bogota Metro Line 2 Project ...,NaN,NaN,IBRD/IDA,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,40095216,2026-04-02T23:05:08Z,Western and Central Africa,Mali,Environmental and Social Review Summary,ML-Mali Electricity Sector Improvement Program...,NaN,NaN,IBRD/IDA,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430132,33996042,NaN,Other,"N/A,World",Report,1W-Gender and Climate Change Nexus -- P178057,"FY17 - Other Public Administration,FY17 - Soci...","FY17 - Gender,FY17 - Climate change,FY17 - Ada...",Advisory Services & Analytics,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
430133,34432874,2025-05-16T00:05:57Z,Other,World,Report,1W-Global Program on Justice and the Rule of L...,NaN,NaN,Advisory Services & Analytics,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
430135,34424952,2025-05-15T19:04:41Z,East Asia and Pacific,Cambodia,Report,KH-Cambodia Regional Corridor and Connectivity...,FY17 - Other Transportation,"FY17 - Trade Facilitation,FY17 - Finance for D...",Advisory Services & Analytics,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
430137,34419290,2025-01-16T23:02:42Z,Europe and Central Asia,Moldova,Country Climate and Development Report,MD-Moldova Country Climate and Development Rep...,Other - Public Administration,"Finance,Develop Financial Markets,Macroeconomi...",Advisory Services & Analytics,Public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
